# Stage 10B — Exploratory/Post-hoc Portfolio Backtest

This notebook preserves the frozen Stage 09 inference lock and causal 15% ZigZag candidate population, applies the existing `started != 0` condition **before daily re-ranking**, and evaluates predeclared liquidity/capacity sensitivities.

This is not a second confirmatory unseen-test result. Stage 10 was already observed.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml


def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.portfolio_backtest_stage10b import (
    canonical_json_sha256,
    file_sha256,
    run_stage10b,
)

print("Repository root:", REPOSITORY_ROOT)

In [ ]:
CONFIG_PATH = REPOSITORY_ROOT / "configs/portfolio_backtest_stage10b.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage10b_config = yaml.safe_load(handle)

print("Stage:", stage10b_config["stage_title"])
print("Exploratory/post-hoc:", stage10b_config["exploratory_status"]["posthoc"])
print("Configuration hash:", canonical_json_sha256(stage10b_config))
print("Requested primary scenario:", stage10b_config["primary_scenario"])
print("Scenario count: 3 liquidity × 3 capacity × 2 structures × 2 exits = 36")

In [ ]:
BASELINE_MANIFEST_PATH = (
    REPOSITORY_ROOT
    / stage10b_config["baseline_stage10"]["manifest_file"]
)
STAGE09_LOCK_PATH = (
    REPOSITORY_ROOT
    / stage10b_config["frozen_stage09"]["inference_lock_file"]
)

baseline_manifest = json.loads(
    BASELINE_MANIFEST_PATH.read_text(encoding="utf-8")
)

print("Baseline Stage 10 commit:", baseline_manifest["git_commit_sha"])
print("Baseline Stage 10 config hash:", baseline_manifest["configuration_hash"])
print("Stage 09 lock SHA:", file_sha256(STAGE09_LOCK_PATH))

assert baseline_manifest["git_commit_sha"] == stage10b_config["baseline_stage10"]["expected_git_commit_sha"]
assert baseline_manifest["configuration_hash"] == stage10b_config["baseline_stage10"]["expected_configuration_hash"]
assert file_sha256(STAGE09_LOCK_PATH) == stage10b_config["frozen_stage09"]["expected_inference_lock_sha256"]

In [ ]:
outputs = run_stage10b(
    repository_root=REPOSITORY_ROOT,
    config=stage10b_config,
    write_outputs=True,
)

In [ ]:
filter_summary = outputs["manifest"]["final_signal_filter"]
pd.DataFrame([filter_summary]).T.rename(columns={0: "value"})

In [ ]:
scenario_summary = outputs["scenario_summary"].copy()
scenario_summary[
    [
        "scenario_id",
        "is_primary_scenario",
        "total_return",
        "cagr",
        "maximum_drawdown",
        "net_profit_factor",
        "net_win_rate",
        "accepted_signals",
        "acceptance_rate",
        "maximum_fraction_of_adv20",
        "maximum_distinct_symbols",
        "maximum_open_lots",
        "maximum_symbol_exposure_fraction",
    ]
].sort_values(
    ["total_return", "net_profit_factor"],
    ascending=[False, False],
    kind="stable",
).reset_index(drop=True)

In [ ]:
primary = scenario_summary.loc[
    scenario_summary["is_primary_scenario"]
]
assert len(primary) == 1
primary.T

In [ ]:
integrity = outputs["integrity_audit"]
assert integrity["entry_constraint_violations"].eq(0).all()
assert integrity["negative_cash_events"].eq(0).all()
assert integrity["protected_lot_nonzero_planned_risk_events"].eq(0).all()
assert integrity["open_lots_after_tail"].eq(0).all()
assert integrity["accepted_signals"].eq(integrity["closed_trades"]).all()
print("All Stage 10B internal integrity checks passed.")

In [ ]:
manifest_path = (
    REPOSITORY_ROOT
    / "results/manifests/10b_exploratory_portfolio_backtest_manifest.json"
)
print("Stage 10B manifest:", manifest_path)
print("Git commit:", outputs["manifest"]["git_commit_sha"])
print("Configuration hash:", outputs["manifest"]["configuration_hash"])
print("Stage 09 inference lock:", outputs["manifest"]["frozen_stage09"]["inference_lock_sha256"])
print("Primary scenario:", outputs["manifest"]["primary_scenario"]["scenario_id"])
print()
print("Next action: independently audit Stage 10B outputs before interpreting any winning exploratory specification.")